In [1]:
from typing_extensions import TypedDict
from typing import List
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model("openai:gpt-5-nano")
# llm = init_chat_model("openai:gpt-4o-mini")

In [2]:
class State(TypedDict):

    dish: str
    ingredients: list[dict]
    recipe_steps: str
    plating_instructions: str


class Ingredient(BaseModel):

    name: str
    quantity: str
    unit: str


class IngredientsOutput(BaseModel):

    ingredients: List[Ingredient]

In [3]:
"""Prompt Chaining"""


def list_ingredients(state: State):
    structured_llm = llm.with_structured_output(IngredientsOutput)  # output form 강제
    response = structured_llm.invoke(
        f"{state['dish']}를 만드는 데 필요한 재료 5-8가지를 나열하세요."
    )
    return {"ingredients": response.ingredients}  # structured output


def create_recipe(state: State):
    response = llm.invoke(
        f"{state['ingredients']} 이 재료들을 사용하여, {state['dish']}를 위한 단계별 요리법을 작성하세요."
    )
    return {"recipe_steps": response.content}  # normal llm


def describe_plating(state: State):
    response = llm.invoke(
        f"{state['recipe_steps']}를 기반으로 {state['dish']}를 어떻게 아름답게 플레이팅할지 설명하세요."
    )
    return {"plating_instructions": response.content}  # normal llm

In [4]:
graph_builder = StateGraph(State)

graph_builder.add_node("list_ingredients", list_ingredients)
graph_builder.add_node("create_recipe", create_recipe)
graph_builder.add_node("describe_plating", describe_plating)

graph_builder.add_edge(START, "list_ingredients")
graph_builder.add_edge("list_ingredients", "create_recipe")
graph_builder.add_edge("create_recipe", "describe_plating")
graph_builder.add_edge("describe_plating", END)

graph = graph_builder.compile()

In [5]:
graph.invoke({"dish": "sandwich"})

{'dish': 'sandwich',
 'ingredients': [Ingredient(name='빵', quantity='2', unit='장'),
  Ingredient(name='터키 슬라이스', quantity='2-3', unit='장'),
  Ingredient(name='치즈 슬라이스', quantity='2', unit='장'),
  Ingredient(name='상추', quantity='2', unit='잎'),
  Ingredient(name='토마토', quantity='2-3', unit='조각'),
  Ingredient(name='마요네즈', quantity='1', unit='큰술')],
 'recipe_steps': '다음 재료로 간단한 샌드위치 만드는 방법입니다.\n\n1. 재료 손질\n- 상추는 씻어 물기를 털고 2장 정도 준비합니다.\n- 토마토는 얇게 2–3조각으로 썰어 둡니다.\n- 빵의 안쪽에 바를 마요네즈 1 큰술은 준비가 끝난 상태로 두세요.\n\n2. 빵 손질 및 마요네즈 바르기\n- 빵 2장을 펼칩니다.\n- 각 빵의 안쪽 면에 고르게 마요네즈를 바릅니다.\n\n3. 속 재료 올리기\n- 한쪽 빵에 터키 슬라이스를 골고루 펼칩니다.\n- 그 위에 치즈 슬라이스를 올려 치즈가 보이도록 배치합니다.\n\n4. 채소 추가\n- 토마토 조각을 치즈 위에 올립니다.\n- 상추 두 잎을 토마토 위에 올려 채소를 마무리합니다.\n\n5. 샌드위치 완성\n- 다른 빵 한 조각으로 덮고 살짝 눌러 형태를 잡습니다.\n\n6. 자르기 및 제공\n- 먹기 좋도록 가로로 반으로, 또는 대각선으로 자릅니다.\n\n추가 팁\n- 바삭하게 즐기고 싶으면 팬에 살짝 구워도 좋습니다. (팬에 기름 없이 말 그대로 구워도 무방합니다.)\n- 원하면 소금 한 꼬집이나 후추를 살짝 추가해도 맛이 더 살아납니다.',
 'plating_instructions': '다음 재료 구성과 과정에 맞춰 샌드위치를 더욱 아름답게 보이도록 플레이팅 팁을 정리합니다

In [ ]:
graph.invoke({"dish": "sandwich"})  # 4o-mini

{'dish': 'sandwich',
 'ingredients': [Ingredient(name='빵 (식빵 또는 번)', quantity='2', unit='조각'),
  Ingredient(name='햄 또는 소세지', quantity='2', unit='조각'),
  Ingredient(name='치즈', quantity='1', unit='조각'),
  Ingredient(name='상추', quantity='1', unit='잎'),
  Ingredient(name='토마토', quantity='2', unit='조각'),
  Ingredient(name='마요네즈', quantity='1', unit='큰술'),
  Ingredient(name='겨자 소스', quantity='1', unit='작은술'),
  Ingredient(name='피클', quantity='2', unit='조각')]}

In [ ]:
graph.invoke({"dish": "sandwich"})  # 5-nano

{'dish': 'sandwich',
 'ingredients': [Ingredient(name='식빵', quantity='2', unit='장'),
  Ingredient(name='햄', quantity='2', unit='조각'),
  Ingredient(name='치즈(슬라이스)', quantity='2', unit='장'),
  Ingredient(name='양상추', quantity='1', unit='장'),
  Ingredient(name='토마토', quantity='2-3', unit='슬라이스'),
  Ingredient(name='마요네즈', quantity='1', unit='큰술')]}